# Matrices

Una **matriz** es una colección ordenada de elementos del mismo tipo organizados en **filas y
columnas**. En el fondo es un **Array de 2 dimensiones**: en vez de acceder con un índice `a[i]`,
accedes con dos, `m[i][j]` (fila `i`, columna `j`).


## Características principales

- **Tamaño fijo:** una vez creada, sus dimensiones no cambian (como un array estático de C, y como
  un `numpy.ndarray`). "Insertar" o "borrar" filas en realidad crea una matriz nueva.
- **Acceso aleatorio:** se llega a cualquier elemento directamente por sus índices (fila, columna),
  en O(1).
- **Almacenamiento contiguo:** los elementos viven en celdas de memoria adyacentes, igual que en un
  array.
- **Homogeneidad:** todos los elementos son del mismo tipo (mismo `dtype`).
- **Dimensionalidad:** pueden ser 1-D (vectores), 2-D (matrices) o N-D.

### El concepto que hay que entender: orden fila-mayor (row-major)

La memoria es **1-D**: es una fila larga de celdas. ¿Cómo metemos algo 2-D ahí? Aplanándolo. En el
orden **fila-mayor** (el que usan C y numpy por defecto) la matriz se guarda **fila tras fila**:

```
matriz 2x3          memoria (contigua, 1-D)
[[10 20 30]   -->   10 20 30 40 50 60
 [40 50 60]]        └fila 0┘ └fila 1┘
```

Con `C` columnas, el elemento `(i, j)` cae en el **índice lineal** `i*C + j`, así que su dirección es:

```
dir(i, j) = base + (i*C + j) * itemsize
```

Es la misma idea que en los arrays (`base + i*itemsize`), solo que primero traducimos las dos
coordenadas a un único índice lineal. Y como es una fórmula de coste fijo, el acceso es **O(1)**.

**Por qué usamos numpy:** guarda los valores en crudo y contiguos (como C), es de tamaño fijo y
expone la dirección, el `itemsize` y —muy útil en 2-D— los `strides`, que nos dicen cuántos bytes
hay que saltar para avanzar una fila o una columna.

In [1]:
# Setup
import sys, time
import numpy as np

# Visualizaciones de la lección (reciben matrices numpy 2-D):
#   ver_memoria · ver_get · ver_set · ver_traverse · ver_insert_fila
#   ver_delete_fila · ver_busqueda_lineal · ver_busqueda_binaria
from visualizations.matrix_viz import (
    ver_memoria, ver_get, ver_set, ver_traverse,
    ver_insert_fila, ver_delete_fila,
    ver_busqueda_lineal, ver_busqueda_binaria,
)

print("numpy", np.__version__)

numpy 2.5.0


## ¿Cómo se almacena una matriz en memoria?

Creamos una matriz `int32` y comprobamos que, por dentro, es un bloque contiguo recorrido en orden
fila-mayor. Fíjate en los `strides`: `(12, 4)` significa "avanzar una fila = 12 bytes (3 columnas ×
4), avanzar una columna = 4 bytes".

In [2]:
M = np.array([[10, 20, 30],
              [40, 50, 60]], dtype=np.int32)

base = M.ctypes.data
R, C = M.shape
print("shape (filas, columnas):", M.shape)
print("dirección base        :", hex(base))
print("itemsize              :", M.itemsize, "bytes")
print("strides (fila, col)   :", M.strides, "-> +1 fila salta", M.strides[0],
      "bytes, +1 columna salta", M.strides[1], "bytes")
print("nbytes                :", M.nbytes, "bytes\n")

for i in range(R):
    for j in range(C):
        lineal = i * C + j
        dir_ = base + i * M.strides[0] + j * M.strides[1]
        print(f"  M[{i}][{j}] = {M[i,j]:>2}  ->  lineal {lineal}  ->  {hex(dir_)}")

shape (filas, columnas): (2, 3)
dirección base        : 0x103ac7e50
itemsize              : 4 bytes
strides (fila, col)   : (12, 4) -> +1 fila salta 12 bytes, +1 columna salta 4 bytes
nbytes                : 24 bytes

  M[0][0] = 10  ->  lineal 0  ->  0x103ac7e50
  M[0][1] = 20  ->  lineal 1  ->  0x103ac7e54
  M[0][2] = 30  ->  lineal 2  ->  0x103ac7e58
  M[1][0] = 40  ->  lineal 3  ->  0x103ac7e5c
  M[1][1] = 50  ->  lineal 4  ->  0x103ac7e60
  M[1][2] = 60  ->  lineal 5  ->  0x103ac7e64


In [3]:
# Los bytes CRUDOS: comprueba que salen fila tras fila (10,20,30,40,50,60)
raw = M.tobytes()
print("bytes de datos:", len(raw), "\n")
valores = [int.from_bytes(raw[k*4:(k+1)*4], "little") for k in range(R*C)]
print("orden en memoria (fila-mayor):", valores)

bytes de datos: 24 

orden en memoria (fila-mayor): [10, 20, 30, 40, 50, 60]


In [13]:
# Visualización: la rejilla lógica y su aplanado contiguo en memoria
ver_memoria(M)

strides=(12, 4) -> avanzar 1 fila = 12 bytes (=3×4); avanzar 1 columna = 4 bytes. Todo contiguo en orden fila-mayor.


## Operaciones y complejidad

### 1. Acceso — O(1)

Se llega a cualquier elemento con la fórmula `dir(i,j) = base + (i*C + j)*itemsize`: una
multiplicación, una suma y una lectura. No depende del tamaño de la matriz, así que es **O(1)** en
tiempo. Y en espacio también O(1): no se reserva nada nuevo, solo se lee en el sitio.

In [5]:
ver_get(M, 1, 2)   # fila 1, columna 2

GET [1][2] = 60
  índice lineal = i*C + j = 1*3 + 2 = 5
  dir = base + (i*C+j)*itemsize = 0x103ac7e50 + 20 = 0x103ac7e64
  -> una fórmula de coste fijo, sin recorrer nada: O(1)


### 2. Asignación (SET) — O(1)

Escribir `m[i][j] = v` calcula la misma dirección y sobreescribe esa celda **en su sitio**, sin
mover nada y sin cambiar el tamaño. O(1) en tiempo y espacio.

In [6]:
ver_set(M, 0, 1, 99)

SET [0][1] = 99 -> se sobreescribe la celda en su sitio, sin mover nada. O(1).


### 3. Inserción y eliminación

- **Al final: O(1) en promedio.** Añadir al final (una fila nueva tras la última) no obliga a mover
  los elementos existentes. Como en los arrays dinámicos, es O(1) amortizado (a veces hay que
  redimensionar y copiar, pero repartido sale constante).
- **En una posición específica: O(n)**, con `n` = número de elementos. Insertar o borrar una fila en
  medio obliga a **desplazar** todas las filas siguientes (arriba o abajo) para abrir o cerrar el
  hueco, y eso son tantos elementos movidos como haya por debajo.

Ojo con el matiz de "tamaño fijo": una matriz numpy no crece en sitio; `np.insert`/`np.delete`
crean una matriz **nueva** y copian. El coste O(n) del desplazamiento se mantiene en cualquier caso.

In [7]:
ver_insert_fila(M, 1, [7, 7, 7])   # inserta una fila en i=1; las de abajo bajan

Insertar fila en i=1 -> 1 filas (3 elementos) se desplazan. O(n). (Añadir al final NO desplaza nada -> O(1) amortizado.)


In [8]:
ver_delete_fila(M, 0)              # borra la fila 0; las de abajo suben

Borrar fila i=0 -> 1 filas (3 elementos) se desplazan. O(n). (Borrar la última NO desplaza -> O(1).)


### 4. Búsqueda

- **Lineal — O(n):** sin ningún orden previo, hay que mirar elemento por elemento hasta encontrarlo.
  En el peor caso (no está, o está al final) recorres los `n` elementos.
- **Binaria — O(log n):** solo si los elementos están **ordenados**. En cada paso miras el elemento
  central y descartas **la mitad** del rango, así que el número de pasos crece como `log2(n)`: 8
  elementos → 3 pasos, 1000 → ~10, un millón → ~20. La binaria necesita orden (por eso se aplica
  sobre una fila ordenada o sobre la matriz aplanada y ordenada), pero a cambio es muchísimo más
  rápida que la lineal.

In [9]:
ver_busqueda_lineal(M, 60)    # recorre en orden fila-mayor hasta hallarlo

60 encontrado en [1][2] tras 6 comparaciones. Peor caso O(n).


In [10]:
# Binaria sobre una secuencia ORDENADA (aquí, la matriz aplanada y ordenada)
ordenado = np.sort(M, axis=None)
print("secuencia ordenada:", ordenado, "\n")
ver_busqueda_binaria(ordenado, 50)

secuencia ordenada: [10 20 30 40 50 60] 



50 encontrado en índice 4 tras 2 pasos (de 6 elementos). O(log n).


In [11]:
# La binaria escala como log n: comparación empírica de nº de pasos
import math
print(f"{'n':>10} | {'lineal (peor)':>13} | {'binaria (pasos)':>15}")
print("-"*46)
for n in [8, 64, 1000, 1_000_000]:
    print(f"{n:>10} | {n:>13} | {math.ceil(math.log2(n+1)):>15}")

         n | lineal (peor) | binaria (pasos)
----------------------------------------------
         8 |             8 |               4
        64 |            64 |               7
      1000 |          1000 |              10
   1000000 |       1000000 |              20


### 5. Recorrido (traverse) — O(n)

Visitar todos los elementos es hacer `n = R*C` accesos O(1), uno por celda, en orden fila-mayor.
El tiempo crece con el número de elementos → **O(n)**. En espacio es O(1) mientras solo leas (un
par de contadores de bucle), sin construir nada nuevo proporcional a `n`.

In [12]:
ver_traverse(M, animar=True)   # la casilla resaltada recorre la matriz en bucle

TRAVERSE -> 6 visitas (2×3) -> O(n) tiempo, O(1) espacio.


## Resumen

| Operación | Tiempo | Por qué |
|---|---|---|
| Acceso (get/set) | O(1) | `dir = base + (i*C + j)·itemsize`, coste fijo; no depende del tamaño |
| Inserción/eliminación al final | O(1) amortizado | no hay que desplazar los elementos existentes |
| Inserción/eliminación en posición | O(n) | desplazar las filas siguientes para abrir/cerrar el hueco |
| Búsqueda lineal | O(n) | mirar elemento por elemento |
| Búsqueda binaria (ordenado) | O(log n) | cada paso descarta la mitad del rango |
| Recorrido | O(n) tiempo / O(1) espacio | `R*C` accesos O(1), una visita por celda |

**Idea central:** una matriz es un array 1-D contiguo recorrido en **orden fila-mayor**. Esa
contigüidad + el tamaño uniforme dan el acceso O(1) por la fórmula `base + (i*C + j)·itemsize`; y,
igual que en los arrays, reestructurar (insertar/borrar en medio) es caro —O(n)— porque obliga a
desplazar elementos.